In [1]:
import os
import json
import re
from pathlib import Path

import pandas as pd

In [2]:
GITHUB_TOKEN = "..."

!git clone https://{GITHUB_TOKEN}@github.com/AnnaFattakhova/psycho_llm.git

Cloning into 'psycho_llm'...
remote: Enumerating objects: 4508, done.
remote: Counting objects: 100% (53/53), done.
remote: Compressing objects: 100% (29/29), done.
remote: Total 4508 (delta 41), reused 24 (delta 24), pack-reused 4455 (from 2)
Receiving objects: 100% (4508/4508), 7.39 MiB | 16.31 MiB/s, done.
Resolving deltas: 100% (1349/1349), done.


In [9]:
INPUT_DIR = "/content/psycho_llm/results/giga_chat"
OUTPUT_DIR = "/content/psycho_llm"

In [40]:
# Очистка данных

import json
import re

def clean_explanation(explanation_obj):
    if explanation_obj is None:
        return ""

    # 1. извлекаем текст
    if isinstance(explanation_obj, dict):
        text = (
            explanation_obj.get("Объяснение")
            or explanation_obj.get("explanation")
            or ""
        )
    else:
        text = explanation_obj

    if not isinstance(text, str):
        text = str(text)

    text = text.strip()

    # 2. если внутри JSON
    try:
        parsed = json.loads(text)
        if isinstance(parsed, dict):
            text = (
                parsed.get("Объяснение")
                or parsed.get("explanation")
                or text
            )
    except:
        pass

    # 3. убираем внешнюю обертку вида: Объяснение": ...
    text = re.sub(
        r'^\s*"?\s*(Объяснение|explanation)\s*"?\s*:\s*',
        '',
        text,
        flags=re.IGNORECASE
    )

    # 4. убираем +, но НЕ трогаем текст
    text = text.replace("+", " ")

    # 5. убираем фигурные скобки
    text = text.replace("{", " ").replace("}", " ")

    # 6. КЛЮЧЕВОЕ:
    # если есть много кусков в кавычках → собираем
    parts = re.findall(r'"([^"]{5,})"', text)

    if parts:
        text = " ".join(parts)

    # 7. убираем мусорные кавычки
    text = text.replace('"""', ' ')
    text = text.replace('" "', ' ')
    text = text.replace('""', ' ')

    # 8. убираем переносы строк
    text = text.replace("\n", " ")

    # 9. убираем одиночные символы типа ) + (
    text = re.sub(r'[()\[\]]+', ' ', text)

    # 10. финальная нормализация
    text = text.strip().strip('"').strip()
    text = re.sub(r'\s+', ' ', text)

    text = re.sub(r'\bОбъяснение\b', '', text, flags=re.IGNORECASE)

    return text

In [41]:
# Фильтруем по шкалам, которые нужно игнорировать

def should_skip_scale(test, scale):
    """
    Возвращает True, если шкалу нужно пропустить
    """

    if not isinstance(scale, str):
        return False

    scale_lower = scale.lower()
    test_lower = str(test).lower()

    # --- ASQ / attachment
    if test_lower in ["attachment", "asq"]:
        if scale_lower in ["доминирующий_тип", "dominant"]:
            return True

    # --- EPI / eysenck
    if test_lower in ["eysenck", "epi"]:
        if scale_lower in ["темперамент", "temperament"]:
            return True

    return False

In [42]:
import numpy as np

def parse_scores(scores, test):
    rows = []

    if not isinstance(scores, dict) or len(scores) == 0:
        return [(test, np.nan, np.nan)]

    test_lower = str(test).lower()

    # одна шкала
    if "score" in scores and len(scores) <= 3:
        val = scores.get("score")
        return [(test, val, scores.get("interpretation", np.nan))]

    # несколько шкал
    for scale, value in scores.items():

        scale_str = str(scale).strip()
        scale_lower = scale_str.lower()

        # 1. error только по scale
        if scale_lower == "error":
            rows.append((scale_str, np.nan, np.nan))
            continue

        # 2. фильтры

        if ("attachment" in test_lower or "asq" in test_lower):
            if scale_lower in ["доминирующий_тип", "dominant"]:
                continue

        if ("eysenck" in test_lower or "epi" in test_lower):
            if scale_lower in ["темперамент", "temperament"]:
                continue

        result = np.nan
        interpretation = np.nan

        # 3. 16PF (надёжно)
        if isinstance(value, dict):

            result = np.nan
            interpretation = np.nan

            for k, v in value.items():
                key = str(k).strip().lower()

                # стен / sten
                if "стен" in key or "sten" in key:
                    result = v

                # уровень / interpretation (на всякий случай)
                if "уровень" in key or "interpretation" in key:
                    interpretation = v

            # fallback для обычных тестов
            if np.isnan(result) and "score" in value:
                result = value.get("score", np.nan)
                interpretation = value.get("interpretation", np.nan)

        # числа
        elif isinstance(value, (int, float)):
            result = value

        # строки
        elif isinstance(value, str):
            interpretation = value

        if not scale_str:
            scale_str = test

        rows.append((scale_str, result, interpretation))

    return rows

In [43]:
files = list(Path(INPUT_DIR).glob("*.json"))

print(f"Найдено файлов: {len(files)}")

# Основной цикл
all_rows = []

for file_path in files:
    try:
        with open(file_path, "r", encoding="utf-8") as f:
            data = json.load(f)

        model = data.get("model", np.nan)
        temperature = str(data.get("temperature", np.nan))
        run = data.get("run", np.nan)
        test = data.get("test", np.nan)

        explanation = clean_explanation(data.get("explanation"))
        if not explanation:
            explanation = np.nan

        scores = data.get("scores", {})
        parsed_scores = parse_scores(scores, test)

        for scale, result, interpretation in parsed_scores:

            if not scale:
                scale = test

            all_rows.append({
                "model": model,
                "temperature": temperature,
                "run": run,
                "test": test,
                "scale": scale,
                "results": result,
                "interpretation": interpretation,
                "explanation": explanation
            })

    except Exception as e:
        print(f"Ошибка в {file_path.name}: {e}")

Найдено файлов: 99


In [44]:
# Dataframe + сохранение

df = pd.DataFrame(all_rows)

# сортировка для удобства
df = df.sort_values(["model", "test", "run"]).reset_index(drop=True)

# сохранить
OUTPUT_PATH = "/content/psycho_dataset.csv"
df.to_csv(OUTPUT_PATH, index=False, encoding="utf-8-sig")

print("Готово:", OUTPUT_PATH)
print(df.shape)
df.head()

Готово: /content/psycho_dataset.csv
(504, 8)


,model,temperature,run,test,scale,results,interpretation,explanation
0,gigachat,0.0,1,16PF,A,6.0,Средний уровень,Я проанализировала предложенные вопросы и вар...
1,gigachat,0.0,1,16PF,B,4.0,Средний уровень,Я проанализировала предложенные вопросы и вар...
2,gigachat,0.0,1,16PF,C,7.0,Средний уровень,Я проанализировала предложенные вопросы и вар...
3,gigachat,0.0,1,16PF,E,7.0,Средний уровень,Я проанализировала предложенные вопросы и вар...
4,gigachat,0.0,1,16PF,F,8.0,Высокий уровень,Я проанализировала предложенные вопросы и вар...
